In [1]:
import os
import duckdb
import pandas as pd
from datasets import Dataset
from huggingface_hub import HfApi

In [5]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())

df = con.sql("SELECT * FROM gold.coco_training").df()
print(f"Rows to push: {len(df)}")
df.head()

Rows to push: 36781


,image_uri,category,bbox_x,bbox_y,bbox_w,bbox_h,split
0,s3://lakehouse/assets/coco/images/139.jpg,dining table,321.21,231.22,446.77,320.15,val
1,s3://lakehouse/assets/coco/images/632.jpg,book,527.02,248.57,551.42,289.00,val
2,s3://lakehouse/assets/coco/images/872.jpg,baseball glove,368.64,157.25,426.09,203.03,val
3,s3://lakehouse/assets/coco/images/1000.jpg,person,265.33,95.86,354.25,411.74,val
4,s3://lakehouse/assets/coco/images/1000.jpg,person,52.14,185.12,111.40,397.65,val


In [6]:
parquet_path = "/data/local/gold_coco_training.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Saved to {parquet_path}")

Saved to /data/local/gold_coco_training.parquet


In [7]:
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "beninod-34/lakehouse-coco-gold"
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True)

ds = Dataset.from_parquet(parquet_path)
ds.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/datasets/{HF_REPO}")

Generating train split: 0 examples [00:00, ? examples/s]

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to https://huggingface.co/datasets/beninod-34/lakehouse-coco-gold


In [8]:
api = HfApi(token=HF_TOKEN)
info = api.dataset_info(HF_REPO)
print(f"Dataset: {info.id}")
print(f"Size: {info.card_data}")

Dataset: beninod-34/lakehouse-coco-gold
Size: dataset_info:
  features:
  - name: image_uri
    dtype: large_string
  - name: category
    dtype: large_string
  - name: bbox_x
    dtype: float64
  - name: bbox_y
    dtype: float64
  - name: bbox_w
    dtype: float64
  - name: bbox_h
    dtype: float64
  - name: split
    dtype: large_string
  splits:
  - name: train
    num_bytes: 4003296
    num_examples: 36781
  download_size: 901372
  dataset_size: 4003296
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*


In [9]:
con.close()